In [1]:
import os
os.chdir("/home/mdarif/MLOPS/Wine-Quality-Prediction")
print(os.getcwd())

/home/mdarif/MLOPS/Wine-Quality-Prediction


In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    source_file: Path
    train_file: Path
    test_file: Path
    scaler_file: Path
    status_file: Path

In [3]:
from wine_quality_prediction.constants import *
from wine_quality_prediction.utils import read_yaml, create_directories

class ConfigurationManager:

    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_fileapath=PARAMS_FILE_PATH,
            schema_filepath=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_fileapath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])
        return DataTransformationConfig(
            root_dir=Path(config.root_dir),
            source_file=Path(config.source_file),
            train_file=Path(config.train_file),
            test_file=Path(config.test_file),
            scaler_file=Path(config.scaler_file),
            status_file=Path(config.status_file)
        )

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from wine_quality_prediction.logger import get_logger, DataTransformationError
from wine_quality_prediction.utils import get_size, save_status, save_bin

logger = get_logger("data_transformation", "data_transformation.log")

In [ ]:
class DataTransformation:
    """Handles data cleaning, scaling and splitting."""

    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def load_data(self) -> pd.DataFrame:
        """Load data from source file."""

        try:
            logger.info(f"Reading file: '{self.config.source_file}'")
            df = pd.read_csv(self.config.source_file)
            logger.info(f"Data loaded successfully - shape: {df.shape}")
            logger.info(f"Columns found: {df.columns.tolist()}")
            return df

        except Exception as e:
            logger.error(
                f"Load data failed - "
                f"path='{self.config.source_file}': {e}",
                exc_info=True
            )
            raise DataTransformationError("Failed to load data") from e

    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Clean column names and prepare dataset ."""

        try:
            df = df.copy()
            df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
            logger.info(f"Columns after normalization: '{df.columns.tolist()}'")
            return df

        except Exception as e:
            logger.error(f"Data cleaning failed: {e}", exc_info=True)
            raise DataTransformationError("Failed to clean data") from e

    def remove_duplicates(self, df: pd.DataFrame) -> pd.DataFrame:
        """Remove duplicates rows from the datasets."""

        try:
            duplicate_count = df.duplicated().sum()
            if duplicate_count == 0:
                logger.info("No duplicate rows found")
                return df
            original_shape = df.shape
            df = df.drop_duplicates().reset_index(drop=True)
            logger.info(
                f"Removed '{duplicate_count}' duplicate rows"
                f"Shape changed from '{original_shape}' to '{df.shape}'"
            )
            return df

        except Exception as e:
            logger.error(f"Duplicate removal failed: {e}", exc_info=True)
            raise DataTransformationError("Failed to remove duplicates rows") from e

    def split_data(self, df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        """Split data into train and test sets."""

        try:
            train, test = train_test_split(
                df,
                test_size=0.2,
                random_state=42
            )
            logger.info(f"Train shape: {train.shape}")
            logger.info(f"Test shape: {test.shape}")
            return train, test

        except Exception as e:
            logger.error(f"Data split failed: {e}", exc_info=True)
            raise DataTransformationError("Failed to split data") from e

    def save_data(self, train: pd.DataFrame, test: pd.DataFrame):
        """Save train and test datasets to CSV files."""

        try:
            train.to_csv(self.config.train_file, index=False)
            test.to_csv(self.config.test_file, index=False)

            logger.info(f"Train file saved: '{self.config.train_file}'")
            logger.info(f"Test file saved: '{self.config.test_file}'")

            logger.info(f"Train file size: '{get_size(self.config.train_file)}'")
            logger.info(f"Test file size: '{get_size(self.config.test_file)}'")

        except Exception as e:
            logger.error(f"Data save failed: {e}", exc_info=True)
            raise DataTransformationError("Failed to save data") from e

    def run(self):
        """Run full data transformation pipeline."""

        try:
            df = self.load_data()
            df = self.clean_data(df)
            df = self.remove_duplicates(df)
            train, test = self.split_data(df)
            self.save_data(train, test)
            save_status(
                self.config.status_file,
                "data_transformation",
                True
            )
            logger.info("Data transformation completed successfully")

        except Exception as e:
            save_status(self.config.status_file, "data_transformation", False)
            logger.error(f"Unexpected error: {e}", exc_info=True)
            raise DataTransformationError("Data transformation pipeline failed") from e

SyntaxError: expected 'except' or 'finally' block (3810451985.py, line 42)

In [6]:
try:
    config = ConfigurationManager()
    data_tranformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_tranformation_config)
    data_transformation.run()
except Exception:
    raise

2026-06-02 08:17:30,427 - common - INFO - common.py:31 - YAML file loaded successfully: 'config/config.yaml'
2026-06-02 08:17:30,430 - common - INFO - common.py:31 - YAML file loaded successfully: 'params.yaml'
2026-06-02 08:17:30,433 - common - INFO - common.py:31 - YAML file loaded successfully: 'schema.yaml'
2026-06-02 08:17:30,434 - common - INFO - common.py:53 - Created directory at: 'artifacts'
2026-06-02 08:17:30,437 - common - INFO - common.py:53 - Created directory at: 'artifacts/data_transformation'
2026-06-02 08:17:30,438 - data_transformation - INFO - 453704506.py:11 - Reading file: 'artifacts/data_ingestion/WineQT.csv'
2026-06-02 08:17:30,445 - data_transformation - INFO - 453704506.py:13 - Data loaded successfully - shape: (1143, 13)
2026-06-02 08:17:30,446 - data_transformation - INFO - 453704506.py:14 - Columns found: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulph